# Milestone 2: Embedding + Vector Database (Simple)

This notebook implements only the Milestone 2 retrieval indexing part:
- Embedding model: `BAAI/bge-large-en-v1.5`
- Vector database: `FAISS`

Pipeline:
1. Load retrieval corpus chunks
2. Generate dense embeddings
3. Build FAISS index
4. Save index + metadata
5. Run a simple semantic retrieval query

In [1]:
# If needed, uncomment and run once in your environment.
!pip install -q sentence-transformers faiss-cpu numpy

In [2]:
from pathlib import Path
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

ROOT = Path('..').resolve()
INPUT_PATH = ROOT / 'data' / 'raw' / 'dataset_v1' / 'retrieval_corpus.json'
OUTPUT_DIR = ROOT / 'data' / 'processed' / 'embedding'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = OUTPUT_DIR / 'faiss_index_ip.bin'
META_PATH = OUTPUT_DIR / 'faiss_metadata.json'

MODEL_NAME = 'BAAI/bge-large-en-v1.5'

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

texts = [item['text'] for item in corpus]
metadata = [
    {
        'chunk_id': item.get('chunk_id'),
        'category': item.get('category'),
        'source_type': item.get('source_type'),
        'text': item.get('text'),
    }
    for item in corpus
]

print(f'Loaded {len(texts)} chunks from: {INPUT_PATH}')

Loaded 1000 chunks from: C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\raw\dataset_v1\retrieval_corpus.json


In [4]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

embeddings = embeddings.astype(np.float32)
dim = embeddings.shape[1]
print(f'Embeddings shape: {embeddings.shape}')

index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f'FAISS index total vectors: {index.ntotal}')

c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\budhi\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2704.22it/s]
Be

Embeddings shape: (1000, 1024)
FAISS index total vectors: 1000


In [5]:
faiss.write_index(index, str(INDEX_PATH))
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Saved files:')
print(f'- {INDEX_PATH}')
print(f'- {META_PATH}')

Saved files:
- C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\embedding\faiss_index_ip.bin
- C:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\data\processed\embedding\faiss_metadata.json


In [6]:
def retrieve(query: str, top_k: int = 5):
    q_emb = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        hit = metadata[idx]
        results.append(
            {
                'score': float(score),
                'chunk_id': hit['chunk_id'],
                'category': hit['category'],
                'source_type': hit['source_type'],
                'text': hit['text'],
            }
        )
    return results

In [7]:
query = 'Avoid using mutable default arguments in Python functions'
results = retrieve(query, top_k=5)

for i, r in enumerate(results, start=1):
    print(f"{i}. score={r['score']:.4f} | chunk_id={r['chunk_id']} | category={r['category']} | source={r['source_type']}")
    print(r['text'][:220].replace('\n', ' ') + '...')
    print('-' * 100)

1. score=0.8839 | chunk_id=chunk_0730 | category=mutable_default | source=augmented
Avoid using mutable default arguments as they can retain changes between function calls. Use `None` and initialize the mutable object inside the function instead....
----------------------------------------------------------------------------------------------------
2. score=0.8744 | chunk_id=chunk_0696 | category=mutable_default | source=augmented
Using a mutable default argument can lead to unintended side effects. Consider using `None` and initializing the mutable object within the function....
----------------------------------------------------------------------------------------------------
3. score=0.8711 | chunk_id=chunk_0657 | category=mutable_default | source=augmented
Avoid using mutable default arguments as they can retain changes across calls, leading to unpredictable behavior. Use `None` and initialize the mutable object within the function....
---------------------------------------------